In [ ]:
import os
import shutil
import importlib.util

local_ragas_path = 'ragas'

spec = importlib.util.find_spec('ragas')
if spec is None or spec.origin is None:
    raise ImportError("Não foi possível encontrar a biblioteca 'ragas'.")

library_ragas_path = os.path.dirname(spec.origin)

if not os.path.exists(library_ragas_path):
    raise FileNotFoundError(f"O diretório da biblioteca 'ragas' não foi encontrado: {library_ragas_path}")

for root, dirs, files in os.walk(local_ragas_path):
    for file in files:
        local_file_path = os.path.join(root, file)
        relative_path = os.path.relpath(local_file_path, local_ragas_path)
        library_file_path = os.path.join(library_ragas_path, relative_path)
        os.makedirs(os.path.dirname(library_file_path), exist_ok=True)
        shutil.copy2(local_file_path, library_file_path)

print("Arquivos copiados com sucesso!")

In [1]:
import pandas as pd

from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from datasets import Dataset
from ragas.integrations.llama_index import evaluate
from ragas.testset.prompts import translate_prompts
from ragas.run_config import RunConfig
from ragas.metrics.critique import SUPPORTED_ASPECTS

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    load_index_from_storage,
)

from ragas.metrics import (
    answer_relevancy,
    answer_correctness,
    answer_similarity,
    context_precision,
    context_recall,
    context_utilization,
    context_entity_recall,
    noise_sensitivity_irrelevant,
    noise_sensitivity_relevant,
    faithfulness
)


In [2]:
DATA_PATH = 'data'
TESTSET = 'testset'
PERSIST_DIR = "./storage"
LANGUAGE = 'pt'
LANGUAGE_CACHE = 'cache'
TIMEOUT = 2400
RESULT_CSV = 'result_2.csv'

metrics = [
    answer_relevancy,
    answer_correctness,
    answer_similarity,
    context_precision,
    context_recall,
    context_utilization,
    context_entity_recall,
    noise_sensitivity_irrelevant,
    noise_sensitivity_relevant,
    faithfulness
]

metrics.extend(SUPPORTED_ASPECTS)


In [3]:
testset = pd.read_csv('testset.csv', index_col=0)
print("Tamanho do dataset: ", len(testset))
print(testset.head())

Tamanho do dataset:  111
                                            question  \
0  Aqui está uma pergunta que pode ser totalmente...   
1  Aqui está uma pergunta que pode ser completame...   
2  Qual é o propósito dos alarmes controlados pel...   
3  Em que condições de umidade relativa o equipam...   
4  Qual é o papel da Equalização Ativa no monitor...   

                                            contexts  \
0  ['ESPECIFICAÇÕES TÉCNICAS \nEntrada\nn Tensão:...   
1  ['\n» Bateria Baixa - um toque a cada 2 segund...   
2  ['n  \nn   \nn   \nn   \nn   \nn   \nn   \nn  ...   
3  [' Acabamento: pintura epóxi-pó na cor preta\n...   
4  ['TECNOLOGIA  EQUALIZER\nEqualização A tiva In...   

                                        ground_truth evolution_type  \
0  A variação admissível da frequência para os eq...         simple   
1  O alarme indica uma bateria baixa com um toque...         simple   
2  Os alarmes controlados pelo processador DSP se...         simple   
3  O equipamento 

In [4]:
nan_rows = testset[testset.isna().any(axis=1)]
print("Quantidade de nulos: ", len(nan_rows))
print(nan_rows)

del nan_rows

Quantidade de nulos:  1
                                              question  \
102  Em situações de ruído eletrônico, o Estabiliza...   

                                              contexts ground_truth  \
102  ['Estabilizador\nEletrônicoDigital Signal Proc...          NaN   

    evolution_type                                           metadata  \
102    conditional  [{'page_label': '1', 'file_name': 'catalogo-pe...   

     episode_done  
102          True  


In [5]:
testset = testset.dropna()

In [6]:
print("Tamanho do dataset: ", len(testset))
print(testset.head())


Tamanho do dataset:  110
                                            question  \
0  Aqui está uma pergunta que pode ser totalmente...   
1  Aqui está uma pergunta que pode ser completame...   
2  Qual é o propósito dos alarmes controlados pel...   
3  Em que condições de umidade relativa o equipam...   
4  Qual é o papel da Equalização Ativa no monitor...   

                                            contexts  \
0  ['ESPECIFICAÇÕES TÉCNICAS \nEntrada\nn Tensão:...   
1  ['\n» Bateria Baixa - um toque a cada 2 segund...   
2  ['n  \nn   \nn   \nn   \nn   \nn   \nn   \nn  ...   
3  [' Acabamento: pintura epóxi-pó na cor preta\n...   
4  ['TECNOLOGIA  EQUALIZER\nEqualização A tiva In...   

                                        ground_truth evolution_type  \
0  A variação admissível da frequência para os eq...         simple   
1  O alarme indica uma bateria baixa com um toque...         simple   
2  Os alarmes controlados pelo processador DSP se...         simple   
3  O equipamento 

In [7]:
testset = Dataset.from_pandas(testset)

In [8]:
translate_prompts(LANGUAGE, LANGUAGE_CACHE)

In [9]:
embeding = OllamaEmbedding(model_name="llama3.1")
model = Ollama(model="llama3.1", request_timeout=TIMEOUT)

Settings.embed_model = embeding
Settings.llm = model

In [10]:
if not os.path.exists(PERSIST_DIR):
    documents = SimpleDirectoryReader(DATA_PATH).load_data()
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

query_engine = index.as_query_engine(request_timeout=TIMEOUT)

In [11]:
result = evaluate(
    query_engine=query_engine,
    metrics=metrics,
    dataset=testset,
    llm=model,
    embeddings=embeding,
    run_config=RunConfig(timeout=TIMEOUT, max_workers=2)
)

Running Query Engine:   0%|          | 0/110 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/880 [00:00<?, ?it/s]

n values greater than 1 not support for LlamaIndex LLMs
n values greater than 1 not support for LlamaIndex LLMs

Failed to parse output. Returning None.


-------------Prompt-------------


Dada uma pergunta, uma resposta e frases da resposta, analise a complexidade de cada frase dada em 'frases' e divida cada frase em uma ou mais declarações totalmente compreensíveis, ao mesmo tempo em que garante que nenhum pronome seja usado em cada declaração. Formate as saídas em JSON.

A saída deve ser uma instância JSON bem formatada que esteja em conformidade com o esquema JSON abaixo.

Como exemplo, para o esquema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
o objeto {"foo": ["bar", "baz"]} é uma instância bem formatada do esquema. O objeto {"properties": {"foo": ["bar", "baz"]}} não está bem formatado.

Aqui está o esquema JSON de saída:
```
{"type": "array", "items": {"$ref": "#/definitions/St

In [12]:
result_dataframe = result.to_pandas()
result_dataframe.to_csv(RESULT_CSV)

In [14]:
print(result)

{'answer_relevancy': 0.5541, 'answer_correctness': 0.8941, 'answer_similarity': 0.5743, 'context_precision': 0.4136, 'context_recall': 0.7810, 'context_utilization': 0.5409, 'faithfulness': nan, 'harmfulness': 0.1009}
